### Comparaison des Modélisations pour la Fréquence et les Coûts des Sinistres

Dans cette section, nous allons comparer différentes approches de modélisation afin d'évaluer deux aspects clés des sinistres :

1. **La fréquence des sinistres** : Modélisation du nombre de sinistres survenus pour chaque contrat.
2. **Les coûts des sinistres** : Estimation des montants associés aux sinistres.

L'objectif est d'identifier les modèles les plus performants pour chaque aspect, en utilisant des métriques d'évaluation adaptées.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import numpy as np
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error
import statsmodels.api as sm
import statsmodels.formula.api as smf

from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_poisson_deviance
import re
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm

import numpy as np
from sklearn.model_selection import KFold
from pyglmnet import GLM


In [2]:
freq = pd.read_parquet("data/raw/freMTPLfreq.parquet")
sev = pd.read_parquet("data/raw/freMTPLsev.parquet")

In [3]:
#Combiner les 2 bases de données

freq['PolicyID'] = freq['PolicyID'].astype(str)
sev['PolicyID'] = sev['PolicyID'].astype(str)

# Agrégation de sev : un contrat peut avoir plusieurs sinistres
sev_agg = (
    sev.groupby("PolicyID")
       .agg(
           ClaimAmount=("ClaimAmount", "sum"),   # coût total des sinistres du contrat
           ClaimNb_sev=("ClaimAmount", "size")   # nombre de sinistres déclarés dans sev
       )
       .reset_index()
)

print(len(sev), "lignes sev brutes")
print(len(sev_agg), "lignes sev agrégées (1 par PolicyID)")

merged = pd.merge(freq, sev_agg, on="PolicyID", how="left")

# Remplacer les NaN (contrats sans sinistre) par 0
merged["ClaimAmount"] = merged["ClaimAmount"].fillna(0)
merged["ClaimNb_sev"] = merged["ClaimNb_sev"].fillna(0)

print("Nb de lignes freq :", len(freq))
print("Nb de lignes merged :", len(merged))


16181 lignes sev brutes
15390 lignes sev agrégées (1 par PolicyID)
Nb de lignes freq : 413169
Nb de lignes merged : 413169


# Création de différentes classes actuarielles nécessaires pour notre modélisation


In [4]:
# 1) Contrôles de base
print(merged[["PolicyID","ClaimNb","Exposure","ClaimAmount"]].head())
print(merged[["ClaimNb","Exposure","ClaimAmount"]].describe())

# 2) Variables actuarielles de base
merged["Freq"] = merged["ClaimNb"] / merged["Exposure"]          # fréquence annuelle
merged["PurePremium"] = merged["ClaimAmount"] / merged["Exposure"]  # prime pure
merged["AvgClaim"] = np.where(                                    # coût moyen conditionnel
    merged["ClaimNb"] > 0,
    merged["ClaimAmount"] / merged["ClaimNb"],
    np.nan
)

# 3) Quelques ratios globaux (à commenter dans le rapport)
tot_expo = merged["Exposure"].sum()
tot_claims = merged["ClaimNb"].sum()
tot_amount = merged["ClaimAmount"].sum()

freq_globale = tot_claims / tot_expo
pure_prem_globale = tot_amount / tot_expo
avgclaim_globale = tot_amount / tot_claims

print("Fréquence globale :", freq_globale)
print("Prime pure globale :", pure_prem_globale)
print("Coût moyen global :", avgclaim_globale)
print(f"% contrats sans sinistre: {(merged['ClaimNb']==0).mean()*100:.1f}%")

merged.columns

  PolicyID  ClaimNb  Exposure  ClaimAmount
0        1        0      0.09          0.0
1        2        0      0.84          0.0
2        3        0      0.52          0.0
3        4        0      0.45          0.0
4        5        0      0.15          0.0
             ClaimNb       Exposure   ClaimAmount
count  413169.000000  413169.000000  4.131690e+05
mean        0.039163       0.561088  8.341642e+01
std         0.204053       0.369477  4.192526e+03
min         0.000000       0.002732  0.000000e+00
25%         0.000000       0.200000  0.000000e+00
50%         0.000000       0.540000  0.000000e+00
75%         0.000000       1.000000  0.000000e+00
max         4.000000       1.990000  2.036833e+06
Fréquence globale : 0.06979858984933181
Prime pure globale : 148.66904231188673
Coût moyen global : 2129.9720042024596
% contrats sans sinistre: 96.3%


Index(['PolicyID', 'ClaimNb', 'Exposure', 'Power', 'CarAge', 'DriverAge',
       'Brand', 'Gas', 'Region', 'Density', 'ClaimAmount', 'ClaimNb_sev',
       'Freq', 'PurePremium', 'AvgClaim'],
      dtype='object')

# Premières approches de modélisation avec loi Poisson pour la fréquence et loi Gamma pour la sévérité

In [5]:

# ============================================================
# 1. Préparation des données et création des classes actuarielles
# ============================================================

# On garde uniquement les contrats exposés
df_glm = merged[merged["Exposure"] > 0].copy()

# Variables de classes si besoin
bins_driver = [17, 25, 30, 40, 50, 60, 120]
labels_driver = ["<25", "25-29", "30-39", "40-49", "50-59", "60+"]
df_glm["DriverAgeClass"] = pd.cut(df_glm["DriverAge"], bins=bins_driver, labels=labels_driver)

bins_car = [-1, 1, 5, 10, 20, 200]
labels_car = ["0", "1-4", "5-9", "10-19", "20+"]
df_glm["CarAgeClass"] = pd.cut(df_glm["CarAge"], bins=bins_car, labels=labels_car)

bins_dens = [0, 50, 200, 500, 2000, 10000]
labels_dens = ["rural", "peri-urbain", "petite ville", "ville", "urbain dense"]
df_glm["DensityClass"] = pd.cut(df_glm["Density"], bins=bins_dens, labels=labels_dens)

# Déclaration des qualitatives
for col in ["Power", "Brand", "Gas", "Region",
            "DriverAgeClass", "CarAgeClass", "DensityClass"]:
    df_glm[col] = df_glm[col].astype("category")

print("Taille base GLM :", len(df_glm))


Taille base GLM : 413169


In [6]:
# =========================
# 2. Split 75 % / 25 %
# =========================

df_train, df_test = train_test_split(df_glm, test_size=0.25, random_state=123)

print("Taille train :", len(df_train))
print("Taille test  :", len(df_test))


Taille train : 309876
Taille test  : 103293


In [7]:
# Base des contrats sinistrés
df_gamma_sev_train = df_train[df_train["ClaimNb"] > 0].copy() #on garde que les contrats avec au moins un sinistre
df_gamma_sev_train["AvgClaim"] = df_gamma_sev_train["ClaimAmount"] / df_gamma_sev_train["ClaimNb"]
print("Taille base sévérité (train) :", len(df_gamma_sev_train))

df_gamma_sev_test = df_test[df_test["ClaimNb"] > 0].copy()
df_gamma_sev_test["AvgClaim"] = df_gamma_sev_test["ClaimAmount"] / df_gamma_sev_test["ClaimNb"]
print("Taille base sévérité (test) :", len(df_gamma_sev_test))

Taille base sévérité (train) : 11511
Taille base sévérité (test) : 3879


## Comparaison entre GLM Poisson et GLM Négative Binomiale Basé sur l'AIC et la deviance (pour la fréquence)

In [8]:
# =========================
# 3. Modèle Poisson complet
# =========================

formula_full = (
    "ClaimNb ~ C(DriverAgeClass) + C(CarAgeClass) "
    "+ C(Power) + C(Region) + C(Gas) + C(DensityClass) + C(Brand)"
)

model_full = smf.glm(
    formula=formula_full,
    data=df_train,
    family=sm.families.Poisson(),
    offset=np.log(df_train["Exposure"])
)
res_pois = model_full.fit()

aic_poi=res_pois.aic
dev_poi=res_pois.deviance
print("AIC Poisson complet :", res_pois.aic)
print("deviance",res_pois.deviance)


# =========================
# 3. Binomiale négative complète (train)
# =========================

nb_family = sm.families.NegativeBinomial()

model_nb = smf.glm(
    formula=formula_full,
    data=df_train,
    family=nb_family,
    offset=np.log(df_train["Exposure"])
)
res_nb = model_nb.fit()
aic_nb=res_nb.aic
dev_nb=res_nb.deviance
print("AIC Binomiale négative :", res_nb.aic)
print("deviance", res_nb.deviance)

AIC Poisson complet : 96237.59914789077
deviance 73891.03017112348


c:\Users\thoma\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\genmod\families\family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


AIC Binomiale négative : 96019.11212487993
deviance 64970.1001454333


In [9]:
#Implémentation stepwise qui permet la sélection des variables selon l'AIC: une variable est ajoutée ou supprimée uniquement si cela diminue l’AIC du modèle courant ; sinon l’algorithme s’arrête.

def stepwise_glm_offset(data, response, candidates, family, offset_var, verbose=True):
    """
    data       : DataFrame
    response   : variable cible
    candidates : liste de termes de formule, ex. ["C(Region)", "C(DensityClass)"]
    family     : famille GLM, ex. sm.families.Poisson()
    offset_var : variable d'exposition
    """
    selected = []
    remaining = candidates.copy()
    current_aic = None
    changed = True

    while changed:
        changed = False

        # ----- FORWARD : ajout de la meilleure variable -----
        best_aic = None
        best_var = None

        for var in remaining:
            terms = selected + [var]
            rhs = " + ".join(terms) if terms else "1"
            formula = f"{response} ~ {rhs}"
            model = smf.glm(
                formula=formula,
                data=data,
                family=family,
                offset=np.log(data[offset_var])
            ).fit()
            aic = model.aic

            if (best_aic is None) or (aic < best_aic):
                best_aic = aic
                best_var = var

        if best_var is not None and (current_aic is None or best_aic < current_aic):
            selected.append(best_var)
            remaining.remove(best_var)
            current_aic = best_aic
            changed = True
            if verbose:
                print(f"Ajout de {best_var}, AIC = {current_aic:.2f}")

        # ----- BACKWARD : retrait de la pire variable -----
        improved = True
        while improved and len(selected) > 1:
            improved = False
            worst_aic = current_aic
            worst_var = None

            for var in selected:
                trial_vars = [v for v in selected if v != var]
                rhs = " + ".join(trial_vars) if trial_vars else "1"
                formula = f"{response} ~ {rhs}"
                model = smf.glm(
                    formula=formula,
                    data=data,
                    family=family,
                    offset=np.log(data[offset_var])
                ).fit()
                aic = model.aic

                if aic < worst_aic:
                    worst_aic = aic
                    worst_var = var

            if worst_var is not None:
                selected.remove(worst_var)
                current_aic = worst_aic
                improved = True
                changed = True
                if verbose:
                    print(f"Retrait de {worst_var}, AIC = {current_aic:.2f}")

    # ----- modèle final -----
    if selected:
        rhs = " + ".join(selected)
    else:
        rhs = "1"

    final_formula = f"{response} ~ {rhs}"
    final_model = smf.glm(
        formula=final_formula,
        data=data,
        family=family,
        offset=np.log(data[offset_var])
    ).fit()

    if verbose:
        print("Formule finale :", final_formula)

    return final_model, selected


In [10]:
df_poisson_freq_test2 = df_test.copy()
df_poisson_freq_train2 = df_train.copy()

candidates = [
    "C(DriverAgeClass)","C(CarAgeClass) ",
    "C(Power)","C(Region)","C(Gas)","(DensityClass)"
]

family = sm.families.Poisson()

model_step_poisson, vars_selected = stepwise_glm_offset(
    data=df_poisson_freq_train2,
    response="ClaimNb",
    candidates=candidates,
    family=family,
    offset_var="Exposure",
    verbose=True
)

print("Variables retenues :", vars_selected)
display(model_step_poisson.summary())


Ajout de (DensityClass), AIC = 97074.75
Ajout de C(DriverAgeClass), AIC = 96494.97
Ajout de C(Gas), AIC = 96370.81
Ajout de C(CarAgeClass) , AIC = 96335.82


KeyboardInterrupt: 

In [ ]:
aic_poisson=model_step_poisson.aic
dev_poisson=model_step_poisson.deviance
print("AIC Poisson :", aic_poisson)
print("deviance", dev_poisson)

AIC Poisson : 96302.78139129127
deviance 73968.21241452395


In [ ]:
candidates = [
    "C(DriverAgeClass)", "C(CarAgeClass)",
    "C(Power)", "C(Region)", "C(Gas)", "C(DensityClass)"
]

# Famille binomiale négative au lieu de Poisson
family = sm.families.NegativeBinomial()

model_step_nb, vars_selected_nb = stepwise_glm_offset(
    data=df_poisson_freq_train2,
    response="ClaimNb",
    candidates=candidates,
    family=family,
    offset_var="Exposure",
    verbose=True
)

print("Variables retenues (NB) :", vars_selected_nb)
display(model_step_nb.summary())

c:\Users\thoma\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\genmod\families\family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


Ajout de C(DensityClass), AIC = 96822.95
Ajout de C(DriverAgeClass), AIC = 96261.05
Ajout de C(Gas), AIC = 96143.73
Ajout de C(CarAgeClass), AIC = 96110.49
Ajout de C(Power), AIC = 96081.05
Formule finale : ClaimNb ~ C(DensityClass) + C(DriverAgeClass) + C(Gas) + C(CarAgeClass) + C(Power)
Variables retenues (NB) : ['C(DensityClass)', 'C(DriverAgeClass)', 'C(Gas)', 'C(CarAgeClass)', 'C(Power)']


<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:                ClaimNb   No. Observations:               296411
Model:                            GLM   Df Residuals:                   296385
Model Family:        NegativeBinomial   Df Model:                           25
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -48015.
Date:                Mon, 08 Dec 2025   Deviance:                       65062.
Time:                        09:21:03   Pearson chi2:                 5.06e+05
No. Iterations:                     7   Pseudo R-squ. (CS):           0.003959
Covariance Type:            nonrobust                                         
===================================================================================================
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept                          -2.2463      0.053    -42.346      0.000      -2.350      -2.142
C(DensityClass)[T.peri-urbain]      0.1362      0.030      4.580      0.000       0.078       0.194
C(DensityClass)[T.petite ville]     0.2567      0.033      7.881      0.000       0.193       0.321
C(DensityClass)[T.ville]            0.4089      0.030     13.440      0.000       0.349       0.469
C(DensityClass)[T.urbain dense]     0.5539      0.032     17.174      0.000       0.491       0.617
C(DriverAgeClass)[T.25-29]         -0.6085      0.043    -13.997      0.000      -0.694      -0.523
C(DriverAgeClass)[T.30-39]         -0.8753      0.037    -23.587      0.000      -0.948      -0.803
C(DriverAgeClass)[T.40-49]         -0.7135      0.036    -19.764      0.000      -0.784      -0.643
C(DriverAgeClass)[T.50-59]         -0.8273      0.038    -21.913      0.000      -0.901      -0.753
C(DriverAgeClass)[T.60+]           -0.8669      0.039    -22.052      0.000      -0.944      -0.790
C(Gas)[T.Regular]                  -0.1905      0.021     -9.145      0.000      -0.231      -0.150
C(CarAgeClass)[T.1-4]               0.0240      0.034      0.704      0.482      -0.043       0.091
C(CarAgeClass)[T.5-9]               0.1113      0.034      3.309      0.001       0.045       0.177
C(CarAgeClass)[T.10-19]             0.0020      0.034      0.060      0.952      -0.065       0.069
C(CarAgeClass)[T.20+]              -0.3515      0.100     -3.532      0.000      -0.547      -0.156
C(Power)[T.e]                       0.1235      0.034      3.657      0.000       0.057       0.190
C(Power)[T.f]                       0.1046      0.033      3.133      0.002       0.039       0.170
C(Power)[T.g]                       0.0968      0.033      2.929      0.003       0.032       0.162
C(Power)[T.h]                       0.1432      0.047      3.032      0.002       0.051       0.236
C(Power)[T.i]                       0.2750      0.052      5.301      0.000       0.173       0.377
C(Power)[T.j]                       0.2288      0.053      4.281      0.000       0.124       0.334
C(Power)[T.k]                       0.3084      0.069      4.479      0.000       0.173       0.443
C(Power)[T.l]                       0.2036      0.100      2.032      0.042       0.007       0.400
C(Power)[T.m]                       0.2245      0.152      1.475      0.140      -0.074       0.523
C(Power)[T.n]                       0.3313      0.173      1.912      0.056      -0.008       0.671
C(Power)[T.o]                       0.1194      0.195      0.612      0.541      -0.263       0.502
===================================================================================================
"""

In [ ]:
aic_nb=model_step_nb.aic
dev_nb=model_step_nb.deviance
print("AIC Poisson :", aic_nb)
print("deviance", dev_nb)

AIC Poisson : 96081.05024076188
deviance 65062.03826131525


In [ ]:
# =========================
# 6. Modèle retenu
# =========================

if (model_step_nb.aic < model_step_poisson.aic) and (model_step_nb.deviance < model_step_poisson.deviance):
    res_best = model_step_nb
    best_name = "Binomiale négative"
else:
    res_best = model_step_poisson
    best_name = "Poisson"

print("Modèle retenu pour la tarification (comparaison Poisson vs NB) :", best_name)

df_best = df_test.copy()

# Fréquence prédite par contrat avec le modèle retenu
df_best["lambda_hat_best"] = res_best.predict(df_best, offset=np.log(df_best["Exposure"]))

freq_obs = df_best["ClaimNb"].sum() / df_best["Exposure"].sum()
freq_hat_nb = df_best["lambda_hat_best"].sum() / df_best["Exposure"].sum()
print(f"Fréquence observée : {freq_obs:.5f}")
print(f"Fréquence prédite (NB) : {freq_hat_nb:.5f}")

Modèle retenu pour la tarification (comparaison Poisson vs NB) : Binomiale négative
Fréquence observée : 0.07064
Fréquence prédite (NB) : 0.06649



### Modèle retenu pour la tarification : Binomiale négative

Le modèle de fréquence basé sur une distribution **Binomiale négative** a été retenu pour la tarification. Ce choix repose sur une comparaison avec le modèle Poisson, où la Binomiale négative s'est avérée plus adaptée pour gérer la sur-dispersion des données (variance supérieure à la moyenne).

- **Fréquence observée** : 0.07064  
    Cette valeur correspond à la fréquence moyenne des sinistres observés dans les données de test.

- **Fréquence prédite (NB)** : 0.06649  
    Cette valeur représente la fréquence moyenne des sinistres prédite par le modèle Binomiale négative. Bien que légèrement inférieure à la fréquence observée, elle reste cohérente avec les données et reflète la capacité du modèle à capturer les tendances globales.
```

## Comparaison entre GLM gamma et GLM lognormal basé sur l'AIC et la deviance 


In [ ]:
#Implémentation stepwise qui permet la sélection des variables selon l'AIC: une variable est ajoutée ou supprimée uniquement si cela diminue l’AIC du modèle courant ; sinon l’algorithme s’arrête.

def stepwise_glm_offset_gamma(data, response, candidates, family, verbose=True):
    """
    data       : DataFrame
    response   : variable cible
    candidates : liste de termes de formule, ex. ["C(Region)", "C(DensityClass)"]
    family     : famille GLM, ex. sm.families.Poisson()
    offset_var : variable d'exposition
    """
    selected = []
    remaining = candidates.copy()
    current_aic = None
    changed = True

    while changed:
        changed = False

        # ----- FORWARD : ajout de la meilleure variable -----
        best_aic = None
        best_var = None

        for var in remaining:
            terms = selected + [var]
            rhs = " + ".join(terms) if terms else "1"
            formula = f"{response} ~ {rhs}"
            model = smf.glm(
                formula=formula,
                data=data,
                family=family
            ).fit()
            aic = model.aic

            if (best_aic is None) or (aic < best_aic):
                best_aic = aic
                best_var = var

        if best_var is not None and (current_aic is None or best_aic < current_aic):
            selected.append(best_var)
            remaining.remove(best_var)
            current_aic = best_aic
            changed = True
            if verbose:
                print(f"Ajout de {best_var}, AIC = {current_aic:.2f}")

        # ----- BACKWARD : retrait de la pire variable -----
        improved = True
        while improved and len(selected) > 1:
            improved = False
            worst_aic = current_aic
            worst_var = None

            for var in selected:
                trial_vars = [v for v in selected if v != var]
                rhs = " + ".join(trial_vars) if trial_vars else "1"
                formula = f"{response} ~ {rhs}"
                model = smf.glm(
                    formula=formula,
                    data=data,
                    family=family
                ).fit()
                aic = model.aic

                if aic < worst_aic:
                    worst_aic = aic
                    worst_var = var

            if worst_var is not None:
                selected.remove(worst_var)
                current_aic = worst_aic
                improved = True
                changed = True
                if verbose:
                    print(f"Retrait de {worst_var}, AIC = {current_aic:.2f}")

    # ----- modèle final -----
    if selected:
        rhs = " + ".join(selected)
    else:
        rhs = "1"

    final_formula = f"{response} ~ {rhs}"
    final_model = smf.glm(
        formula=final_formula,
        data=data,
        family=family
    ).fit()

    if verbose:
        print("Formule finale :", final_formula)

    return final_model, selected



In [ ]:
df_sev_train_ln = df_gamma_sev_train.copy()
df_sev_test_ln  = df_gamma_sev_test.copy()



candidates_sev = [
    "C(DriverAgeClass)", "C(CarAgeClass)",
    "C(Power)", "C(Region)", "C(Gas)", "C(DensityClass)"
]

family_gamma = sm.families.Gamma(sm.families.links.log())

model_gamma, vars_gamma = stepwise_glm_offset_gamma(
    data=df_gamma_sev_train,        # DataFrame de sévérité (contrats sinistrés)
    response="AvgClaim",   # sévérité moyenne par sinistre ou par contrat
    candidates=candidates_sev,
    family=family_gamma,
    verbose=True
)


print("Variables Gamma retenues :", vars_gamma)
print("AIC Gamma :", model_gamma.aic)
display(model_gamma.summary())

c:\Users\thoma\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\genmod\families\links.py:13: FutureWarning: The log link alias is deprecated. Use Log instead. The log link alias will be removed after the 0.15.0 release.
  warnings.warn(


Ajout de C(DriverAgeClass), AIC = 238189.02
Ajout de C(DensityClass), AIC = 225900.28
Ajout de C(Power), AIC = 219956.57
Ajout de C(Region), AIC = 219208.78
Ajout de C(CarAgeClass), AIC = 219051.65
Ajout de C(Gas), AIC = 219030.44
Formule finale : AvgClaim ~ C(DriverAgeClass) + C(DensityClass) + C(Power) + C(Region) + C(CarAgeClass) + C(Gas)
Variables Gamma retenues : ['C(DriverAgeClass)', 'C(DensityClass)', 'C(Power)', 'C(Region)', 'C(CarAgeClass)', 'C(Gas)']
AIC Gamma : 219030.43990804342


<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:               AvgClaim   No. Observations:                10974
Model:                            GLM   Df Residuals:                    10939
Model Family:                   Gamma   Df Model:                           34
Link Function:                    log   Scale:                          18.414
Method:                          IRLS   Log-Likelihood:            -1.0948e+05
Date:                Mon, 08 Dec 2025   Deviance:                       16002.
Time:                        09:21:20   Pearson chi2:                 2.01e+05
No. Iterations:                    45   Pseudo R-squ. (CS):           0.009448
Covariance Type:            nonrobust                                         
===================================================================================================
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept                           8.2573      0.274     30.176      0.000       7.721       8.794
C(DriverAgeClass)[T.25-29]         -0.7509      0.187     -4.006      0.000      -1.118      -0.384
C(DriverAgeClass)[T.30-39]         -0.7030      0.160     -4.404      0.000      -1.016      -0.390
C(DriverAgeClass)[T.40-49]         -0.8224      0.155     -5.311      0.000      -1.126      -0.519
C(DriverAgeClass)[T.50-59]         -0.8447      0.162     -5.202      0.000      -1.163      -0.526
C(DriverAgeClass)[T.60+]           -0.6461      0.169     -3.814      0.000      -0.978      -0.314
C(DensityClass)[T.peri-urbain]     -0.0055      0.130     -0.043      0.966      -0.260       0.249
C(DensityClass)[T.petite ville]    -0.0630      0.143     -0.441      0.659      -0.343       0.217
C(DensityClass)[T.ville]           -0.0625      0.136     -0.460      0.645      -0.329       0.204
C(DensityClass)[T.urbain dense]    -0.1279      0.153     -0.838      0.402      -0.427       0.171
C(Power)[T.e]                       0.0350      0.146      0.240      0.811      -0.251       0.321
C(Power)[T.f]                       0.1170      0.144      0.812      0.417      -0.165       0.399
C(Power)[T.g]                       0.1562      0.142      1.101      0.271      -0.122       0.434
C(Power)[T.h]                      -0.1708      0.203     -0.843      0.399      -0.568       0.226
C(Power)[T.i]                       0.7487      0.222      3.380      0.001       0.315       1.183
C(Power)[T.j]                      -0.0243      0.229     -0.106      0.916      -0.474       0.425
C(Power)[T.k]                       0.0398      0.293      0.136      0.892      -0.533       0.613
C(Power)[T.l]                       0.2684      0.429      0.626      0.532      -0.573       1.110
C(Power)[T.m]                       0.7207      0.644      1.119      0.263      -0.541       1.983
C(Power)[T.n]                      -0.6846      0.756     -0.906      0.365      -2.166       0.797
C(Power)[T.o]                      -0.1963      0.835     -0.235      0.814      -1.833       1.440
C(Region)[T.Basse-Normandie]        0.2584      0.288      0.896      0.370      -0.307       0.824
C(Region)[T.Bretagne]               0.0994      0.196      0.506      0.613      -0.285       0.484
C(Region)[T.Centre]                 0.0932      0.170      0.547      0.585      -0.241       0.427
C(Region)[T.Haute-Normandie]       -0.1735      0.388     -0.447      0.655      -0.934       0.587
C(Region)[T.Ile-de-France]          0.0295      0.201      0.147      0.883      -0.365       0.424
C(Region)[T.Limousin]              -0.1939      0.410     -0.473      0.636      -0.998       0.610
C(Region)[T.Nord-Pas-de-Calais]    -0.0278      0.232     -0.120      0.905      -0

In [ ]:
aic_g=model_gamma.aic
dev_g=model_gamma.deviance
print("AIC Gamma :", aic_g)
print("deviance", dev_g)

AIC Gamma : 219030.43990804342
deviance 16001.546530687665


In [ ]:
def stepwise_ols(data, response, candidates, verbose=True):
    selected = []
    remaining = candidates.copy()
    current_aic = None
    changed = True

    while changed:
        changed = False

        # FORWARD
        best_aic = None
        best_var = None
        for var in remaining:
            terms = selected + [var]
            rhs = " + ".join(terms) if terms else "1"
            formula = f"{response} ~ {rhs}"
            model = smf.ols(formula=formula, data=data).fit()
            aic = model.aic
            if (best_aic is None) or (aic < best_aic):
                best_aic = aic
                best_var = var

        if best_var is not None and (current_aic is None or best_aic < current_aic):
            selected.append(best_var)
            remaining.remove(best_var)
            current_aic = best_aic
            changed = True
            if verbose:
                print(f"Ajout de {best_var}, AIC = {current_aic:.2f}")

        # BACKWARD
        improved = True
        while improved and len(selected) > 1:
            improved = False
            worst_aic = current_aic
            worst_var = None
            for var in selected:
                trial = [v for v in selected if v != var]
                rhs = " + ".join(trial) if trial else "1"
                formula = f"{response} ~ {rhs}"
                model = smf.ols(formula=formula, data=data).fit()
                aic = model.aic
                if aic < worst_aic:
                    worst_aic = aic
                    worst_var = var
            if worst_var is not None:
                selected.remove(worst_var)
                current_aic = worst_aic
                improved = True
                changed = True
                if verbose:
                    print(f"Retrait de {worst_var}, AIC = {current_aic:.2f}")

    rhs = " + ".join(selected) if selected else "1"
    final_formula = f"{response} ~ {rhs}"
    final_model = smf.ols(formula=final_formula, data=data).fit()
    if verbose:
        print("Formule finale lognormal :", final_formula)
    return final_model, selected



In [ ]:
df_sev_train_ln["logAvg"] = np.log(df_sev_train_ln["AvgClaim"])
df_sev_test_ln["logAvg"]  = np.log(df_sev_test_ln["AvgClaim"])


candidates_sev = [
    "C(DriverAgeClass)", "C(CarAgeClass)",
    "C(Power)", "C(Region)", "C(Gas)", "C(DensityClass)"
]


model_logn, vars_logn = stepwise_ols(
    data=df_sev_train_ln,
    response="logAvg",
    candidates=candidates_sev,
    verbose=True
)

print("Variables lognormales retenues :", vars_logn)
print("AIC lognormal (train) :", model_logn.aic)
display(model_logn.summary())

Ajout de C(DensityClass), AIC = 33299.74
Ajout de C(DriverAgeClass), AIC = 33279.96
Ajout de C(Region), AIC = 33275.26
Formule finale lognormal : logAvg ~ C(DensityClass) + C(DriverAgeClass) + C(Region)
Variables lognormales retenues : ['C(DensityClass)', 'C(DriverAgeClass)', 'C(Region)']
AIC lognormal (train) : 33275.258432621355


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 logAvg   R-squared:                       0.005
Model:                            OLS   Adj. R-squared:                  0.004
Method:                 Least Squares   F-statistic:                     3.221
Date:                Mon, 08 Dec 2025   Prob (F-statistic):           4.49e-06
Time:                        09:21:21   Log-Likelihood:                -16619.
No. Observations:               10974   AIC:                         3.328e+04
Df Residuals:                   10955   BIC:                         3.341e+04
Df Model:                          18                                         
Covariance Type:            nonrobust                                         
===================================================================================================
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept                           7.0036      0.057    122.541      0.000       6.892       7.116
C(DensityClass)[T.peri-urbain]     -0.0250      0.033     -0.751      0.453      -0.090       0.040
C(DensityClass)[T.petite ville]    -0.0511      0.037     -1.400      0.161      -0.123       0.020
C(DensityClass)[T.ville]           -0.0099      0.035     -0.286      0.775      -0.078       0.058
C(DensityClass)[T.urbain dense]    -0.0100      0.039     -0.258      0.797      -0.086       0.066
C(DriverAgeClass)[T.25-29]         -0.1595      0.048     -3.330      0.001      -0.253      -0.066
C(DriverAgeClass)[T.30-39]         -0.1564      0.041     -3.849      0.000      -0.236      -0.077
C(DriverAgeClass)[T.40-49]         -0.0878      0.039     -2.224      0.026      -0.165      -0.010
C(DriverAgeClass)[T.50-59]         -0.0908      0.041     -2.197      0.028      -0.172      -0.010
C(DriverAgeClass)[T.60+]            0.0048      0.043      0.111      0.912      -0.080       0.089
C(Region)[T.Basse-Normandie]       -0.0517      0.074     -0.699      0.485      -0.197       0.093
C(Region)[T.Bretagne]              -0.0225      0.050     -0.448      0.654      -0.121       0.076
C(Region)[T.Centre]                -0.1029      0.044     -2.365      0.018      -0.188      -0.018
C(Region)[T.Haute-Normandie]       -0.0223      0.099     -0.224      0.823      -0.217       0.173
C(Region)[T.Ile-de-France]          0.0533      0.051      1.037      0.300      -0.048       0.154
C(Region)[T.Limousin]              -0.0372      0.105     -0.354      0.723      -0.243       0.169
C(Region)[T.Nord-Pas-de-Calais]    -0.0216      0.059     -0.363      0.717      -0.138       0.095
C(Region)[T.Pays-de-la-Loire]      -0.1022      0.052     -1.971      0.049      -0.204      -0.001
C(Region)[T.Poitou-Charentes]      -0.0717      0.061     -1.166      0.243      -0.192       0.049
==============================================================================
Omnibus:                     1432.229   Durbin-Watson:                   1.999
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             6033.274
Skew:                          -0.590   Prob(JB):                         0.00
Kurtosis:                       6.435   Cond. No.                         16.9
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [ ]:

# =========================
# 4. Déviance Gamma sur le test (critère commun)
# =========================
print("AIC Gamma :", aic_g)
print("AIC lognormal (train) :", model_logn.aic)

def gamma_deviance(mu, y):
    """Déviance Gamma (scale=1) pour vecteurs numpy positifs."""
    eps = 1e-10
    y = np.asarray(y, float)
    mu = np.asarray(mu, float)
    mask = (y > 0) & (mu > 0) & ~np.isnan(mu)
    y = y[mask]
    mu = mu[mask]
    return 2 * np.sum((y - mu) / mu - np.log(y / mu + eps)), mask.sum()

# Prédictions Gamma
mu_gamma_test = model_gamma.predict(df_sev_test_ln)
dev_gamma, n_g = gamma_deviance(mu_gamma_test, df_sev_test_ln["AvgClaim"])
print(f"Déviance Gamma (test, n={n_g}) :", dev_gamma)

# Prédictions Lognormal ramenées au niveau moyen (E[Y] ≈ exp(m + s²/2])
mu_logn_test_ln = model_logn.predict(df_sev_test_ln)
sigma2 = model_logn.scale  # variance résiduelle sur log
mu_logn_test = np.exp(mu_logn_test_ln + 0.5 * sigma2)

dev_logn, n_l = gamma_deviance(mu_logn_test, df_sev_test_ln["AvgClaim"])
print(f"Déviance (critère Gamma) Lognormal (test, n={n_l}) :", dev_logn)

# =========================
# 5. Modèle de sévérité retenu
# =========================

if (model_logn.aic < model_gamma.aic) and (dev_logn < dev_gamma):
    sev_best = "Lognormal"
else:
    sev_best = "Gamma"

print("Modèle retenu pour la sévérité :", sev_best)


AIC Gamma : 219030.43990804342
AIC lognormal (train) : 33275.258432621355
Déviance Gamma (test, n=3703) : 6643.985463481791
Déviance (critère Gamma) Lognormal (test, n=3703) : 6432.920269925575
Modèle retenu pour la sévérité : Lognormal


### Choix du Modèle Lognormal pour la Sévérité

Après comparaison des performances des modèles Gamma et Lognormal pour la modélisation de la sévérité, le modèle **Lognormal** a été retenu. Voici les principales raisons de ce choix :

1. **AIC (Akaike Information Criterion)** :  
    - Le modèle Lognormal présente un AIC significativement plus faible que celui du modèle Gamma, indiquant une meilleure adéquation aux données d'entraînement.



2. **Performance globale** :  
    - Le modèle Lognormal capture mieux les relations dans les données, ce qui en fait un choix plus robuste pour la modélisation de la sévérité.

En conclusion, le modèle Lognormal est plus performant et sera utilisé pour estimer les coûts moyens conditionnels des sinistres.


In [ ]:
res_sev_ln = model_logn #on recupere le modèle lognormal calculé plus tôt


print(res_sev_ln.summary())
print("AIC Lognormal sévérité (train) :", res_sev_ln.aic)

# Sévérité prédite (espérance lognormale) sur train / test
sigma2 = res_sev_ln.scale

df_sev_train_ln["logAvg_hat"] = res_sev_ln.predict(df_sev_train_ln)
df_sev_test_ln["logAvg_hat"]  = res_sev_ln.predict(df_sev_test_ln)

df_sev_train_ln["sev_hat"] = np.exp(df_sev_train_ln["logAvg_hat"] + 0.5 * sigma2)
df_sev_test_ln["sev_hat"]  = np.exp(df_sev_test_ln["logAvg_hat"]  + 0.5 * sigma2)



print(df_sev_test_ln["sev_hat"] .mean())

                            OLS Regression Results                            
Dep. Variable:                 logAvg   R-squared:                       0.005
Model:                            OLS   Adj. R-squared:                  0.004
Method:                 Least Squares   F-statistic:                     3.221
Date:                Mon, 08 Dec 2025   Prob (F-statistic):           4.49e-06
Time:                        09:21:21   Log-Likelihood:                -16619.
No. Observations:               10974   AIC:                         3.328e+04
Df Residuals:                   10955   BIC:                         3.341e+04
Df Model:                          18                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept 

In [ ]:
#on rajoute les frequences estimées au dataframe de train et de test

df_train["lambda_hat"] = model_step_nb.predict(df_train, offset=np.log(df_train["Exposure"]))
df_test["lambda_hat"]  = model_step_nb.predict(df_test,  offset=np.log(df_test["Exposure"]))

# Rattacher sev_hat aux dataframes fréquence
df_train = df_train.merge(
    df_sev_train_ln[["PolicyID", "sev_hat"]],
    on="PolicyID",
    how="left"
)
df_test = df_test.merge(
    df_sev_test_ln[["PolicyID", "sev_hat"]],
    on="PolicyID",
    how="left"
)

# Tarification & validation

In [ ]:
# Pour les contrats sans sinistre (pas de prédiction directe), on peut
# utiliser la sévérité moyenne globale estimée sur sev_train
sev_global = df_train["sev_hat"].mean()
df_train["sev_hat"] = df_train["sev_hat"].fillna(sev_global)
df_test["sev_hat"]  = df_test["sev_hat"].fillna(sev_global)

# ============================================================
# 4. Prime pure modélisée et tarif
# ============================================================

# Prime pure modélisée par contrat (unité d'exposition)
df_train["PurePremium_hat"] = df_train["lambda_hat"] * df_train["sev_hat"]
df_test["PurePremium_hat"]  = df_test["lambda_hat"]  * df_test["sev_hat"]

# Chargement global (ex : +20 %)
loading = 1.20
df_train["Tarif"] = df_train["PurePremium_hat"] * loading
df_test["Tarif"]  = df_test["PurePremium_hat"]  * loading

# ============================================================
# 5. Vérification du tarif sur l’échantillon de validation
#    -> comparaison prime pure observée vs modélisée par segment
# ============================================================

group_cols = ["DriverAgeClass", "Power", "Region"]

check_test = (
    df_test
    .groupby(group_cols)
    .agg(
        Exposure_tot=("Exposure", "sum"),
        Pure_obs=("PurePremium", "mean"),
        Pure_hat=("PurePremium_hat", "mean"),
        Tarif_moy=("Tarif", "mean")
    )
    .reset_index()
)

print("Vérification sur l'échantillon de validation (quelques segments) :")
display(check_test.sort_values("Exposure_tot", ascending=False).head(15))

# Comparaison globale sur le test
pure_obs_globale = (df_test["ClaimAmount"].sum() / df_test["Exposure"].sum())
pure_hat_globale = (df_test["PurePremium_hat"].sum() / df_test["Exposure"].sum())

print("Prime pure observée (test)  :", pure_obs_globale)
print("Prime pure modélisée (test) :", pure_hat_globale)

# Ratio modèle / observé
print("Ratio modèle / observé :", pure_hat_globale / pure_obs_globale)

Vérification sur l'échantillon de validation (quelques segments) :


C:\Users\thoma\AppData\Local\Temp\ipykernel_75828\1791397941.py:29: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(group_cols)


,DriverAgeClass,Power,Region,Exposure_tot,Pure_obs,Pure_hat,Tarif_moy
383,40-49,f,Centre,1608.721288,323.353863,76.238508,91.486209
263,30-39,f,Centre,1553.444164,305.425258,61.677823,74.013388
393,40-49,g,Centre,1473.794849,182.326372,71.023893,85.228672
623,60+,f,Centre,1446.532909,502.597644,74.459012,89.350815
513,50-59,g,Centre,1372.538418,214.761272,67.946292,81.535550
503,50-59,f,Centre,1306.773798,753.431430,69.252609,83.103130
633,60+,g,Centre,1272.075609,77.940247,69.016772,82.820127
273,30-39,g,Centre,1265.293205,148.026016,54.957077,65.948493
373,40-49,e,Centre,1123.480222,99.148069,77.295212,92.754255
253,30-39,e,Centre,1116.862079,1096.095959,64.263420,77.116104


Prime pure observée (test)  : 148.27044660402692
Prime pure modélisée (test) : 114.32340407842014
Ratio modèle / observé : 0.7710464674307874


## mise en place d'un GLM poisson avec lasso pour la selection de variables


In [45]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import PoissonRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import make_scorer, mean_poisson_deviance
import numpy as np


# =========================
# 1. Préparation des données
# =========================

# Les données sont déjà préparées dans X_encoded, y, et sample_weight
# X_encoded : variables explicatives (one-hot encodées)
# y : fréquence (ClaimNb / Exposure)
# sample_weight : poids (Exposure)

# =========================
# 2. Validation croisée pour alpha
# =========================

df = df_glm.copy()
df = df[df["Exposure"] > 0].copy()

# Variable réponse = fréquence
y = df["ClaimNb"] / df["Exposure"]

# Variables explicatives (toutes qualitatives ici)
X_cat = df[["DriverAgeClass", "CarAgeClass", "Power", "Region", "Gas", "DensityClass"]].astype("category")

# One-hot encoding sans drop de la catégorie de base (scikit gère la pénalisation)
enc = OneHotEncoder(drop=None, sparse_output=False)
X_encoded = enc.fit_transform(X_cat)

# Poids = exposition (offset en GLM)
sample_weight = df["Exposure"].to_numpy()

# Split 75 / 25
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X_encoded, y, sample_weight, test_size=0.25, random_state=123
)


# Définir une grille de valeurs pour alpha
alpha_grid = np.logspace(-4, 1, 50)

# Définir le modèle Poisson avec Lasso
model = PoissonRegressor(fit_intercept=True, max_iter=1000)

# Définir la validation croisée
cv = KFold(n_splits=5, shuffle=True, random_state=123)

# Définir le score basé sur la déviance de Poisson
scorer = make_scorer(mean_poisson_deviance, greater_is_better=False, needs_proba=True)

# Configurer la recherche de grille
grid_search = GridSearchCV(
    estimator=model,
    param_grid={'alpha': alpha_grid},
    scoring=scorer,
    cv=cv,
    n_jobs=-1
)

# Ajuster le modèle
grid_search.fit(X_encoded, y, sample_weight=sample_weight)

# Meilleur alpha
best_alpha = grid_search.best_params_['alpha']
print("Meilleur alpha (Lasso) :", best_alpha)

# =========================
# 3. Modèle final avec le meilleur alpha
# =========================

model_lasso = PoissonRegressor( alpha=best_alpha, fit_intercept=True, max_iter=1000)
model_lasso.fit(X_encoded, y, sample_weight=sample_weight)

# Récupérer les coefficients non nuls avec leurs variables
coef = model_lasso.coef_
feature_names = enc.get_feature_names_out(X_cat.columns)
# Calculate AIC and Deviance for the model
# Calcul correct de l'AIC pour le GLM Poisson scikit-learn
n = X_encoded.shape[0]
k = np.count_nonzero(coef) + 1  # nombre de paramètres non nuls + intercept
ll = -mean_poisson_deviance(y, model_lasso.predict(X_encoded), sample_weight=sample_weight) * n / 2  # log-vraisemblance approx
aic = 2 * k - 2 * ll
deviance = mean_poisson_deviance(y, model_lasso.predict(X_encoded), sample_weight=sample_weight)

print(f"AIC of the model: {aic:.2f}")
print(f"Deviance of the model: {deviance:.2f}")

selected = [(name, c) for name, c in zip(feature_names, coef) if abs(c) > 1e-6]
selected = sorted(selected, key=lambda x: x[1], reverse=True)

print("Variables sélectionnées par le Lasso (coefs non nuls) :")
for name, c in selected:
    print(f"{name:35s}  coef={c:.3f}")

c:\Users\thoma\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_search.py:1108: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(


Meilleur alpha (Lasso) : 0.0001
AIC of the model: 185303.38
Deviance of the model: 0.45
Variables sélectionnées par le Lasso (coefs non nuls) :
DriverAgeClass_<25                   coef=0.633
DensityClass_nan                     coef=0.332
DensityClass_urbain dense            coef=0.214
Region_Limousin                      coef=0.160
CarAgeClass_5-9                      coef=0.159
Power_k                              coef=0.144
Gas_Diesel                           coef=0.121
Power_i                              coef=0.103
DensityClass_ville                   coef=0.087
CarAgeClass_1-4                      coef=0.081
Region_Poitou-Charentes              coef=0.062
Power_j                              coef=0.050
Power_n                              coef=0.044
CarAgeClass_0                        coef=0.043
Power_m                              coef=0.040
CarAgeClass_10-19                    coef=0.037
Region_Aquitaine                     coef=0.030
DriverAgeClass_25-29                 coe

In [72]:
import numpy as np
from sklearn.model_selection import KFold
from pyglmnet import GLM



df = df_glm.copy()
df = df[df["Exposure"] > 0].copy()

# Variable réponse = fréquence
y = df["ClaimNb"] / df["Exposure"]

# Variables explicatives (toutes qualitatives ici)
X_cat = df[["DriverAgeClass", "CarAgeClass", "Power", "Region", "Gas", "DensityClass"]].astype("category")

# One-hot encoding sans drop de la catégorie de base (scikit gère la pénalisation)
enc = OneHotEncoder(drop=None, sparse_output=False)
X_encoded = enc.fit_transform(X_cat)

# Poids = exposition (offset en GLM)
sample_weight = df["Exposure"].to_numpy()

# Split 75 / 25
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X_encoded, y, sample_weight, test_size=0.25, random_state=123
)

X_train=np.array(X_train)
print(X_train.shape)
y_train=np.array(y_train)
print(y_train.shape)




# grille de lambda (force de pénalisation)
lambdas = np.linspace(0, 1e-3, 20)

kf = KFold(n_splits=5, shuffle=True, random_state=0)

mean_scores = []

best_score = None
for lam in lambdas:
    cv_scores = []
    print("Lambda =", lam)
    for train_idx, val_idx in kf.split(X_train):
        X_train_fold, X_val = X_train[train_idx], X_train[val_idx]
        y_train_fold, y_val = y_train[train_idx], y_train[val_idx]

        glm = GLM(
            distr='poisson',
            alpha=1.0,
                          # Lasso pur
            reg_lambda=lam,
            max_iter=1000,
            learning_rate=1e-3,
            tol=1e-5
        )
        glm.fit(X_train_fold, y_train_fold)
        cv_scores.append(glm.score(X_val, y_val))

    mean_cv = np.mean(cv_scores)
    mean_scores.append(mean_cv)
    print("Score moyen CV =", mean_cv)

    if best_score is None or mean_cv < best_score:
        best_score = mean_cv
    else:
        print("Score non amélioré, arrêt du grid search.")
        break

best_idx = np.argmin(mean_scores)
best_lambda = lambdas[best_idx]
print("Meilleur lambda =", best_lambda)

(309876, 41)
(309876,)
Lambda = 0.0


c:\Users\thoma\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyglmnet\pyglmnet.py:863: UserWarning: Reached max number of iterations without convergence.
  warnings.warn(


Score moyen CV = 76750.69701780796
Lambda = 5.2631578947368424e-05
Score moyen CV = 82133.78632072957
Score non amélioré, arrêt du grid search.
Meilleur lambda = 0.0


In [74]:
X_train=np.array(X_train)
y_train=np.array(y_train)


glm_final = GLM(
    distr='poisson',
    alpha=1.0,               # Lasso pur
    reg_lambda=best_lambda,
    max_iter=1000,
    learning_rate=1e-3,
    tol=1e-5
)

# Fit sur tout l'échantillon d'apprentissage
glm_final.fit(X_train, y_train)


# Prédictions sur le train
lasso_pred = glm_final.predict(X_train)

# Prédictions sur le test
X_test = np.array(X_test)
y_test = np.array(y_test)
y_test_pred = glm_final.predict(X_test)

# =========================

train_score = glm_final.score(X_train, y_train)   
test_score = glm_final.score(X_test, y_test)

print(f"deviance (train) : {train_score:.4f}")
print(f"deviance (test)  : {test_score:.4f}")

deviance (train) : 383700.4639
deviance (test)  : 124886.8646


In [75]:
print("Intercept :", glm_final.beta0_)
print("Coefficients :", glm_final.beta_)
print("Nb coefs non nuls :", np.count_nonzero(glm_final.beta_))
print("Pred unique ?", np.allclose(y_test_pred, y_test_pred[0]))

Intercept : -0.4120661999401744
Coefficients : [-0.03307231 -0.09085234 -0.06113718 -0.04902103 -0.09341427  0.00343341
 -0.08396246 -0.12805231 -0.11475866 -0.00417941 -0.08134462 -0.05570615
 -0.07960913 -0.0906848  -0.09291115  0.00296103 -0.02498376 -0.01347392
 -0.0316788  -0.06542398  0.01323152  0.01886644 -0.01962992  0.01518634
 -0.0477477  -0.0475686  -0.16778529  0.02495284 -0.04525585 -0.00192894
 -0.02157485 -0.06267621 -0.06774945 -0.23387316 -0.22475531 -0.08279385
 -0.04264965 -0.09796269 -0.0857644  -0.10841404 -0.05375733]
Nb coefs non nuls : 41
Pred unique ? False


In [80]:

# Calculate AIC and Deviance for the model
# Calcul correct de l'AIC pour le GLM Poisson scikit-learn
n = X_train.shape[0]
k = np.count_nonzero(coef) + 1  # nombre de paramètres non nuls + intercept
ll = -mean_poisson_deviance(y, glm_final.predict(X_encoded), sample_weight=sample_weight) * n / 2  # log-vraisemblance approx
aic = 2 * k - 2 * ll
deviance = mean_poisson_deviance(y, glm_final.predict(X_encoded), sample_weight=sample_weight)
print(f"AIC of the model: {aic:.2f}")


AIC of the model: 243677.73


In [78]:
# 4) Fréquence prédite et contrôle global
#lasso_pred = model_lasso.predict(X_test)

freq_obs = df_test["ClaimNb"].sum() / df_test["Exposure"].sum()
freq_hat_nb = y_test_pred.sum() / df_test["Exposure"].sum()
print(f"Fréquence observée : {freq_obs:.5f}")
print(f"Fréquence prédite (lasso) : {freq_hat_nb:.5f}")

Fréquence observée : 0.07064
Fréquence prédite (lasso) : 0.62630


In [47]:

# Calculate the mean of the product of lasso_pred and sev_hat
result = (lasso_pred * df_test["sev_hat"]).mean()
print(result)

KeyError: 'sev_hat'

## Approche avec une Quasi-Poisson

In [ ]:
# On part de df_glm déjà préparé (Exposure > 0, variables catégorielles, etc.)

formula_freq = (
    "ClaimNb ~ C(DriverAgeClass) + C(CarAgeClass) "
    "+ C(Power) + C(Region) + C(Gas) + C(DensityClass)"
)

# Estimation Poisson avec ajustement de dispersion (quasi-Poisson)
model_qp = smf.glm(
    formula=formula_freq,
    data=df_glm,
    family=sm.families.Poisson(),
    offset=np.log(df_glm["Exposure"])
)

result_qp = model_qp.fit(scale="X2")   # ou scale="pearson"
print(result_qp.summary())

# Les prédictions de fréquence restent les mêmes que le Poisson simple
df_glm["lambda_hat_qp"] = result_qp.predict(df_glm, offset=np.log(df_glm["Exposure"]))

                 Generalized Linear Model Regression Results                  
Dep. Variable:                ClaimNb   No. Observations:               395215
Model:                            GLM   Df Residuals:                   395180
Model Family:                 Poisson   Df Model:                           34
Link Function:                    Log   Scale:                          1.7431
Method:                          IRLS   Log-Likelihood:                -36972.
Date:                Mon, 08 Dec 2025   Deviance:                       99098.
Time:                        09:22:30   Pearson chi2:                 6.89e+05
No. Iterations:                     9   Pseudo R-squ. (CS):           0.002349
Covariance Type:            nonrobust                                         
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept 